# 01 -- pypdf : lecture et manipulation

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- ouvrir un fichier PDF avec `pypdf` ;
- extraire le texte d'un PDF ;
- acceder aux metadonnees (titre, auteur) ;
- fusionner plusieurs PDF ;
- extraire et reordonner des pages ;
- chiffrer et dechiffrer un PDF ;
- faire pivoter et decouper des pages ;
- gerer les PDF avec annotations.

## Prerequis -- ce que vous devez maitriser

- fonctions typees, `pathlib` ;
- context managers.

Ce que ce notebook **ne couvrira pas** :

- l'extraction precise de tableaux (voir `02_Pdfplumber.ipynb`) ;
- la creation riche (voir `04_Reportlab_platypus.ipynb`) ;
- l'OCR (voir `03_Pymupdf.ipynb`).

## Plan

1. Installation
2. Lecture simple
3. Extraction de texte
4. Metadonnees
5. Fusion
6. Extraction de pages
7. Rotation
8. Decoupage
9. Chiffrement
10. Annotations
11. Pages paires/impaires
12. Synthese
13. Exercices

---

## 1. Installation

In [ ]:
# pip install pypdf
import pypdf
pypdf.__version__

---

## 2. Lecture simple

In [ ]:
# Pour les demos, creons un PDF simple avec reportlab
# (en pratique, vous ouvrez un existant)
from reportlab.pdfgen import canvas
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
pdf_path = tmp / 'demo.pdf'

c_pdf = canvas.Canvas(str(pdf_path))
c_pdf.drawString(100, 750, 'Page 1 - Titre')
c_pdf.drawString(100, 730, 'Ligne de contenu')
c_pdf.showPage()
c_pdf.drawString(100, 750, 'Page 2 - Suite')
c_pdf.save()
print('PDF demo cree')

In [ ]:
from pypdf import PdfReader
reader = PdfReader(str(pdf_path))
print('Nb pages :', len(reader.pages))

---

## 3. Extraction de texte

In [ ]:
for i, page in enumerate(reader.pages):
    print(f'--- Page {i+1} ---')
    print(page.extract_text())

---

## 4. Metadonnees

In [ ]:
from pypdf import PdfWriter

writer = PdfWriter()
writer.append_pages_from_reader(reader)
writer.add_metadata({
    '/Title': 'Mon document',
    '/Author': 'Alice',
    '/Subject': 'Demo pypdf',
    '/Keywords': 'python, pdf, demo',
})
meta_path = tmp / 'with_meta.pdf'
with open(meta_path, 'wb') as f:
    writer.write(f)

r2 = PdfReader(str(meta_path))
print(r2.metadata)

---

## 5. Fusion

In [ ]:
# Creer deux PDF
p1 = tmp / 'p1.pdf'
p2 = tmp / 'p2.pdf'
for path, label in [(p1, 'Doc 1'), (p2, 'Doc 2')]:
    cc = canvas.Canvas(str(path))
    cc.drawString(100, 750, label)
    cc.save()

# Fusionner
from pypdf import PdfWriter
writer = PdfWriter()
for p in [p1, p2]:
    writer.append(str(p))
merged = tmp / 'merged.pdf'
with open(merged, 'wb') as f:
    writer.write(f)
print('Pages apres fusion :', len(PdfReader(str(merged)).pages))

---

## 6. Extraction de pages

In [ ]:
reader = PdfReader(str(merged))
writer = PdfWriter()
writer.add_page(reader.pages[0])  # n'ajoute que la premiere
out = tmp / 'page1_only.pdf'
with open(out, 'wb') as f:
    writer.write(f)
print('Extrait page 1')

---

## 7. Rotation

In [ ]:
reader = PdfReader(str(pdf_path))
writer = PdfWriter()
for page in reader.pages:
    page.rotate(90)
    writer.add_page(page)
with open(tmp / 'rotated.pdf', 'wb') as f:
    writer.write(f)
print('Rotations appliquees')

---

## 8. Decoupage

In [ ]:
# Splitter un PDF en plusieurs fichiers de 1 page chacun
reader = PdfReader(str(merged))
for i, page in enumerate(reader.pages):
    w = PdfWriter()
    w.add_page(page)
    with open(tmp / f'split_{i+1}.pdf', 'wb') as f:
        w.write(f)
print('Splitte en', len(reader.pages), 'fichiers')

---

## 9. Chiffrement

In [ ]:
writer = PdfWriter()
writer.append(str(pdf_path))
writer.encrypt('secret')
enc_path = tmp / 'encrypted.pdf'
with open(enc_path, 'wb') as f:
    writer.write(f)
print('PDF chiffre')

In [ ]:
# Lecture d'un PDF chiffre
r = PdfReader(str(enc_path))
print('Chiffre ?', r.is_encrypted)
r.decrypt('secret')
print('Texte :', r.pages[0].extract_text()[:50])

---

## 10. Annotations

In [ ]:
# Lister les annotations d'une page
reader = PdfReader(str(pdf_path))
page = reader.pages[0]
if '/Annots' in page:
    for annot in page['/Annots']:
        print(annot.get_object())
else:
    print('Aucune annotation')

---

## 11. Pages paires/impaires

In [ ]:
reader = PdfReader(str(merged))
odd = PdfWriter()
even = PdfWriter()
for i, p in enumerate(reader.pages):
    (odd if i % 2 == 0 else even).add_page(p)
with open(tmp / 'odd.pdf', 'wb') as f:
    odd.write(f)
with open(tmp / 'even.pdf', 'wb') as f:
    even.write(f)
print('Odd et even separes')

---

## Synthese

| Operation | pypdf |
|---|---|
| Lire | `PdfReader(path)` |
| Ecrire | `PdfWriter()` + `.write(f)` |
| Extraire texte | `page.extract_text()` |
| Metadonnees | `reader.metadata` / `writer.add_metadata(...)` |
| Fusion | `writer.append(path)` |
| Rotation | `page.rotate(degrees)` |
| Split | `w = PdfWriter(); w.add_page(p)` |
| Chiffrer | `writer.encrypt(password)` |

### Regles a retenir

- **pypdf** = extraction texte correcte mais pas parfaite sur PDF complexes.
- **`extract_text`** peut rater l'ordre des blocs : preferer pdfplumber pour les PDF structures.
- **`encrypt`** : RC4 ou AES selon `use_rc4=`.
- **Pages pypdf indexees a partir de 0**.

---

## Exercices

### Exercice 1 (facile) -- Lire un PDF

Fonction `count_pages(path)` qui retourne le nombre de pages.

In [ ]:
from pypdf import PdfReader

# TODO

<details>
<summary>Voir la correction</summary>

```python
from pypdf import PdfReader

def count_pages(path: str) -> int:
    return len(PdfReader(path).pages)
```

</details>

### Exercice 2 (facile) -- Extraire tout le texte

Fonction `extract_all_text(path) -> str` qui concatene le texte de toutes les pages.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pypdf import PdfReader

def extract_all_text(path: str) -> str:
    reader = PdfReader(path)
    return '\n'.join(p.extract_text() or '' for p in reader.pages)
```

</details>

### Exercice 3 (moyen) -- Merge

Fonction `merge_pdfs(paths: list[str], output: str)` qui fusionne.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pypdf import PdfWriter

def merge_pdfs(paths: list[str], output: str) -> None:
    writer = PdfWriter()
    for p in paths:
        writer.append(p)
    with open(output, 'wb') as f:
        writer.write(f)
```

</details>

### Exercice 4 (moyen) -- Extraire une plage

Fonction `extract_range(input, output, start, end)` qui prend les pages `[start, end)` indexees a 1.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pypdf import PdfReader, PdfWriter

def extract_range(input_path: str, output_path: str, start: int, end: int) -> None:
    reader = PdfReader(input_path)
    writer = PdfWriter()
    for i in range(start - 1, end - 1):
        writer.add_page(reader.pages[i])
    with open(output_path, 'wb') as f:
        writer.write(f)
```

</details>

### Exercice 5 (difficile) -- Secure merge

Fonction `secure_merge(paths, output, password)` qui fusionne et chiffre le resultat.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pypdf import PdfWriter

def secure_merge(paths: list[str], output: str, password: str) -> None:
    writer = PdfWriter()
    for p in paths:
        writer.append(p)
    writer.encrypt(password)
    with open(output, 'wb') as f:
        writer.write(f)
```

</details>

---

## Ressources externes

- [pypdf documentation](https://pypdf.readthedocs.io/)
- [PDF Reference](https://www.adobe.com/content/dam/acom/en/devnet/pdf/pdf_reference_1-7.pdf)